In [ ]:
import os
import logging

# Enable basic debugging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# --- Use Ollama locally ---
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

# Initialize the local LLM (e.g., gemma3:1b or mistral)
llm = ChatOllama(model="gemma3:1b", temperature=0)
logger.info("Initialized local LLM via Ollama: gemma3:1b")

# Initialize HuggingFace embeddings (you can choose another model)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
logger.info("Initialized HuggingFace embeddings")

# Initialize vector store
vector_store = InMemoryVectorStore(embeddings)
logger.info("Initialized in-memory vector store")

# --- Load & Split Website Document ---
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
logger.info(f"Loading web content from: {url}")

loader = WebBaseLoader(
    web_paths=(url,),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))
    ),
)

docs = loader.load()
logger.info(f"Loaded {len(docs)} raw documents")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)
logger.info(f"Split into {len(all_splits)} document chunks")

# --- Embed & Store Vectors ---
vector_store.add_documents(documents=all_splits)
logger.info("Indexed document chunks")

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
logger.info("Created retriever")

# --- Build RAG Chain ---
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain.prompts import ChatPromptTemplate

SYSTEM = """You are an expert assistant.
Answer *only* from the context between <context></context>;
if the answer isn’t there, say “I don't know.”"""
USER = """<context>\n{context}\n</context>\n\nQuestion: {input}"""

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM), ("user", USER)])

combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)
logger.info("Assembled RAG chain")

In [2]:
# --- Query ---
question = "What is Task Decomposition?"
logger.info(f"Asking question: {question}")

result = rag_chain.invoke({"input": question})

print("\n=== Answer ===\n", result["answer"])
print("\n=== Context ===\n", result["context"])


INFO:__main__:Asking question: What is Task Decomposition?



=== Answer ===
 Chain of thought (CoT) has become a standard prompting technique for enhancing model performance on complex tasks. It transforms big tasks into multiple manageable tasks and sheds light into an interpretation of the model’s thinking process.

=== Context ===
 [Document(id='e440cf45-01bb-43e8-a96f-5a0e7ad260e7', metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Fig. 1. Overview of a LLM-powered autonomous agent system.\nComponent One: Planning#\nA complicated task usually involves many steps. An agent needs to know what they are and plan ahead.\nTask Decomposition#\nChain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an interpretation o